### Dataset implementation

In [1]:
import os

import torch
from torch.utils.data import Dataset
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

In [2]:
def process_csv(file_path):
    df = pd.read_csv(file_path)

    images_dict = {}
    
    for i in range(len(df)):
        id_full = df.iloc[i]['ID']
        label_value = df.iloc[i]['Label']
        
        # split the image's ID from the hemorrhage's type
        # ex: "ID_xxx_epidural" -> "ID_xxx" and "epidural"
        parts = id_full.split('_')
        hemorrhage_type = parts[-1]
        image_id = '_'.join(parts[:-1])
        
        # if the image is not in the dictionary, add it
        if image_id not in images_dict:
            images_dict[image_id] = {}
        
        # add label for the corresponding hemorrhage's type
        images_dict[image_id][hemorrhage_type] = label_value
    
    return images_dict

In [3]:
class RSNAHemorrhageDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.images_dict = process_csv(csv_file)
        self.image_ids = list(self.images_dict.keys())
        self.label_columns = ['epidural', 'intraparenchymal', 'intraventricular',
                             'subarachnoid', 'subdural', 'any']

        print(f"Dataset initialized with {len(self)} images from {img_dir}")
        print(f"Categories: {self.label_columns}")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        # get image's ID
        img_id = self.image_ids[idx]

        # image's path
        img_name = img_id + '_frame0.png'
        img_path = os.path.join(self.img_dir, img_name)

        # load image
        image = Image.open(img_path)
    
        # apply transformations if they exist
        if self.transform:
            image = self.transform(image)

        # extract labels for the image
        labels = []
        for label_col in self.label_columns:
            label_value = self.images_dict[img_id].get(label_col, 0)
            labels.append(label_value)

        # convert labels to tensor
        label = torch.tensor(labels, dtype=torch.float32)

        return image, label

    def get_image_id(self, idx):
        return self.image_ids[idx]

In [ ]:
train_dataset = RSNAHemorrhageDataset(
    csv_file='subdataset_train.csv',
    img_dir='rsna-intracranial-hemorrhage-detection-png/train_images'
)

#### Test

In [5]:
def test_getitem(train_dataset, test_indices, text_file):
    with open(text_file, 'w') as f:
        for idx in test_indices:
            image, label = train_dataset[idx]
            img_id = train_dataset.get_image_id(idx)
            
            f.write(f"Image's name: {img_id}_frame0.png\n")
            f.write(f"Label: {label}\n")
            f.write("\n")

def show_images(train_dataset, test_indices, save_path_image):
    fig, axes = plt.subplots(1, len(test_indices), figsize=(15, 4))

    for i, idx in enumerate(test_indices):
        image, label = train_dataset[idx]
        img_id = train_dataset.get_image_id(idx)
        
        axes[i].imshow(image, cmap='gray')
        axes[i].axis('off')
        axes[i].set_title(f"Index {idx}\n{img_id}", fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path_image, bbox_inches='tight', dpi=300)
    plt.close()

In [6]:
indices = [0, 1, 5, 10, 15, 20, 25]

os.makedirs('output/text', exist_ok=True)
os.makedirs('output/image', exist_ok=True)

test_getitem(train_dataset, indices, text_file='output/text/test_getitem_output.txt')
show_images(train_dataset, indices, save_path_image='output/image/images_visualization.png')

### Train/val split

In [7]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

In [8]:
def split_dataset(dataset, test_size, random_state):
    indices = list(range(len(dataset)))

    train_indices, val_indices = train_test_split(
        indices,
        test_size=test_size,
        random_state=random_state
    )

    train_subset = Subset(dataset, train_indices)
    val_subset = Subset(dataset, val_indices)

    return train_subset, val_subset

In [ ]:
train_subset, val_subset = split_dataset(train_dataset, test_size=0.2, random_state=42)

print(f"Train size: {len(train_subset)}")
train_loader = DataLoader(train_subset, batch_size=16, shuffle=True, num_workers=2)
print(f"Train loader: {len(train_loader)} batch-uri")

print()

print(f"Val size: {len(val_subset)}")
val_loader = DataLoader(val_subset, batch_size=16, shuffle=False, num_workers=2)
print(f"Val loader: {len(val_loader)} batch-uri")

### Class distribution analysis

In [10]:
import numpy as np

In [11]:
def calculate_class_distribution(dataset, indices, label_columns):
    label_counts = {}
    
    for label in label_columns:
        label_counts[label] = 0

    for idx in indices:
        _, label = dataset[idx]
        for i, label_name in enumerate(label_columns):
            if label[i] == 1:
                label_counts[label_name] += 1
    
    return label_counts

In [ ]:
test_dataset = RSNAHemorrhageDataset(
    csv_file='subdataset_test.csv',
    img_dir='rsna-intracranial-hemorrhage-detection-png/test_images'
)

In [13]:
def compute_distributions(train_dataset, test_dataset):
    train_subset, val_subset = split_dataset(train_dataset, test_size=0.2, random_state=42)

    train_distribution = calculate_class_distribution(
        train_dataset,
        train_subset.indices,
        train_dataset.label_columns
    )

    val_distribution = calculate_class_distribution(
        train_dataset,
        val_subset.indices,
        train_dataset.label_columns
    )

    test_distribution = calculate_class_distribution(
        test_dataset,
        list(range(len(test_dataset))),
        test_dataset.label_columns
    )

    return train_distribution, val_distribution, test_distribution

In [14]:
def plot_class_distribution(distributions, title):
    labels = list(distributions[0].keys())
    x = np.arange(len(labels))
    width = 0.25

    fig, ax = plt.subplots(figsize=(10, 6))

    for i, (dist, set_name) in enumerate(zip(distributions, ['Train', 'Validation', 'Test'])):
        counts = []

        for label in labels:
            counts.append(dist[label])

        ax.bar(x + i * width, counts, width, label=set_name)

    ax.set_ylabel('No of instances')
    ax.set_title(title)
    ax.set_xticks(x + width)
    ax.set_xticklabels(labels)
    ax.legend()

    plt.tight_layout()
    plt.savefig('output/image/class_distribution.png', bbox_inches='tight', dpi=300)
    plt.close()

In [15]:
distributions = compute_distributions(train_dataset, test_dataset)

os.makedirs('output/image', exist_ok=True)

plot_class_distribution(
    distributions,
    'Distribution of Hemorrhage Classes in the Train, Validation, and Test Sets'
)

### Visual analysis of hemorrhage types

In [16]:
def find_images_with_single_hemorrhage(dataset, category_name, num_images=8):
    image_ids = []
    
    label_index = dataset.label_columns.index(category_name)
    
    hemorrhage_indices = []
    for i, col in enumerate(dataset.label_columns):
        if col != 'any':
            hemorrhage_indices.append(i)
    
    for idx in range(len(dataset)):
        _, label = dataset[idx]
        
        if label[label_index] == 1:
            hemorrhage_sum = sum(label[i] for i in hemorrhage_indices) # sum of hemorrhage labels
            
            if hemorrhage_sum == 1: # only the specified type is present
                image_ids.append(dataset.get_image_id(idx))
            
                if len(image_ids) >= num_images:
                    break
    
    return image_ids

In [17]:
def extract_single_hemorrhage_images(dataset, num_images=8):
    category_images = {}
    
    hemorrhage_types = []
    for col in dataset.label_columns:
        if col != 'any':
            hemorrhage_types.append(col)
    
    for category in hemorrhage_types:
        image_ids = find_images_with_single_hemorrhage(dataset, category, num_images)
        category_images[category] = image_ids
    
    return category_images

In [18]:
def show_single_hemorrhage_images(category_images, train_dataset):
    for category, image_ids in category_images.items():
            
        fig, axes = plt.subplots(1, len(image_ids), figsize=(20, 3))
        fig.suptitle(f'Images with {category} hemorrhage', fontsize=16)
        
        if len(image_ids) == 1: # only one image found
            axes = [axes]
        
        for i, img_id in enumerate(image_ids):
            idx = train_dataset.image_ids.index(img_id)
            image, label = train_dataset[idx]
            
            axes[i].imshow(image, cmap='gray')
            axes[i].axis('off')
            axes[i].set_title(f"{img_id}", fontsize=9)
        
        plt.tight_layout()
        plt.savefig(f'output/image/hemorrhages/{category}_images.png', 
                   bbox_inches='tight', dpi=300)
        plt.close()

In [19]:
os.makedirs('output/image/hemorrhages', exist_ok=True)

category_images = extract_single_hemorrhage_images(train_dataset, num_images=8)
show_single_hemorrhage_images(category_images, train_dataset)

### Data consistency checks

#### No of channels

In [20]:
from collections import Counter

In [21]:
def check_no_channels_images(dataset, start_idx, end_idx):
    channels_images_grayscale = []
    channels_images_rgb = []
    
    for idx in range(start_idx, end_idx):
        img_id = dataset.image_ids[idx]
        img_path = os.path.join(dataset.img_dir, img_id + '_frame0.png')
        
        img = Image.open(img_path)

        if img.mode == 'L':
            channels_images_grayscale.append(1)
        elif img.mode == 'RGB':
            channels_images_rgb.append(3)
    
    return channels_images_grayscale + channels_images_rgb

In [22]:
def print_channel_dirstribution(dataset):
    no_channles = check_no_channels_images(dataset, 0, len(dataset))
    channels_count = Counter(no_channles)

    for channels, count in channels_count.items():
        print(f"{channels} channels: {count} images")

In [ ]:
print("Channels from train dataset before conversion:")
print_channel_dirstribution(train_dataset)

print()

print("Channels from test dataset before conversion:")
print_channel_dirstribution(test_dataset)

#### Image's dimensions

In [24]:
from PIL import Image
from collections import Counter
import os
import matplotlib.pyplot as plt

In [25]:
def analyze_dataset_sizes(dataset):
    size_counts = Counter()
    size_examples = {}
    
    for idx in range(len(dataset)):
        img_id = dataset.image_ids[idx]
        img_path = os.path.join(dataset.img_dir, f'{img_id}_frame0.png')
        
        with Image.open(img_path) as img:
            size = img.size
            size_counts[size] += 1
            if size not in size_examples:
                size_examples[size] = img_path
    
    return size_counts, size_examples

In [26]:
def plot_dataset_sizes(size_counts, save_path):
    sizes = list(size_counts.keys())
    counts = list(size_counts.values())
    
    size_labels = [f"{w}x{h}" for w, h in sizes]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    bars = ax.bar(range(len(sizes)), counts, color='steelblue', alpha=0.7)
    ax.set_xlabel('Image size', fontsize=12)
    ax.set_ylabel('No', fontsize=12)
    ax.set_title('Distribution Image Sizes', fontsize=14, fontweight='bold')
    ax.set_xticks(range(len(sizes)))
    ax.set_xticklabels(size_labels, rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    
    for i, (bar, count) in enumerate(zip(bars, counts)):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                str(count), ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close()

In [27]:
train_size_counts, train_size_examples = analyze_dataset_sizes(train_dataset)
plot_dataset_sizes(train_size_counts, 'output/image/plot_train_image_sizes.png')

test_size_counts, test_size_examples = analyze_dataset_sizes(test_dataset)
plot_dataset_sizes(test_size_counts, 'output/image/plot_test_image_sizes.png')

In [28]:
def plot_images_by_size(dataset, save_path):
    size_counts, size_examples = analyze_dataset_sizes(dataset)
    
    unique_sizes = list(size_examples.keys())
    
    fig, axes = plt.subplots(1, len(unique_sizes), figsize=(20, 5))
    
    if len(unique_sizes) == 1:
        axes = [axes]
    
    for i, size in enumerate(unique_sizes):
        img_path = size_examples[size]
        with Image.open(img_path) as img:
            axes[i].imshow(img, cmap='gray')
            axes[i].set_title(f"Dimension: {size}", fontsize=10)
            axes[i].axis('off')
    
    plt.tight_layout()
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.close()

In [29]:
analyze_dataset_sizes(train_dataset)
plot_images_by_size(train_dataset, save_path='output/image/image_sizes_distribution_train.png')

analyze_dataset_sizes(test_dataset)
plot_images_by_size(test_dataset, save_path='output/image/image_sizes_distribution_test.png')

#### Pixels values

In [30]:
import numpy as np

In [31]:
def check_pixel_ranges(dataset, output_file):
    ranges = []
    
    for idx in range(len(dataset)):
        img_id = dataset.image_ids[idx]
        img_path = os.path.join(dataset.img_dir, f'{img_id}_frame0.png')
        
        with Image.open(img_path) as img:
            img_array = np.array(img)
            pixel_min = img_array.min()
            pixel_max = img_array.max()
            ranges.append((pixel_min, pixel_max))
    
    range_counts = Counter(ranges)
    
    all_mins = [r[0] for r in ranges]
    all_maxs = [r[1] for r in ranges]
    
    with open(output_file, 'w') as f:
        f.write("Distribution value pixels (min, max):\n")
        for pixel_range, count in sorted(range_counts.items()):
            f.write(f"{pixel_range}: {count} images\n")
        
        f.write("\n")
        f.write(f"Global min value: {min(all_mins)}\n")
        f.write(f"Global max value: {max(all_maxs)}\n")

In [32]:
check_pixel_ranges(train_dataset, output_file='output/text/pixel_ranges_train.txt')
check_pixel_ranges(test_dataset, output_file='output/text/pixel_ranges_test.txt')

### Preprocessing operations

In [33]:
from monai.transforms import GaussianSmooth, GaussianSharpen, MedianSmooth
from skimage import filters
import cv2
from monai.transforms import (
    RandAdjustContrast,
    RandGaussianSmooth,
    RandFlip,
    RandZoom,
    Resize,
    RandRotate
)

In [81]:
class CTPreprocessingPipeline:
    def __init__(self):
        self.techniques = {
            'scale_intensity': self.scale_intensity,
            'resize': self.resize,
            'gaussian_smooth': self.gaussian_smooth,
            'clahe': self.clahe,
            'sobel_edges': self.sobel_edges,
            'gaussian_sharpen': self.gaussian_sharpen,
            'median_filter': self.median_filter,
            'histogram_equalization': self.histogram_equalization, 
            'rand_adjust_contrast': self.rand_adjust_contrast, 
            'rand_gaussian_smooth': self.rand_gaussian_smooth,
            'rand_flip': self.rand_flip, 
            'rand_zoom': self.rand_zoom,
            'rand_rotate': self.rand_rotate
        }

    def scale_intensity(self, image_array):  
        img_min = image_array.min()
        img_max = image_array.max()
        scaled_image = (image_array - img_min) / (img_max - img_min) * 255.0
        return scaled_image.astype(np.float32)
    
    def resize(self, image_array, new_size=(128, 128)):  
        resized_image = cv2.resize(image_array, new_size, interpolation=cv2.INTER_LINEAR)
        return resized_image

    def gaussian_smooth(self, image_array):  
        gaussian_smooth = GaussianSmooth(sigma=5.0)
        return gaussian_smooth(image_array)
    
    def clahe(self, image_array):  
        image_array = image_array.astype(np.uint8)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16, 16))
        return clahe.apply(image_array)

    def sobel_edges(self, image_array):  
        edges = filters.sobel(image_array)
        return edges

    def gaussian_sharpen(self, image_array):  
        gaussian_sharpen = GaussianSharpen(alpha=30.0)
        return gaussian_sharpen(image_array)

    def median_filter(self, image_array):
        median_smooth = MedianSmooth(radius=1)
        return median_smooth(image_array)
    
    def histogram_equalization(self, image_array):
        image_uint8 = image_array.astype(np.uint8)
        return cv2.equalizeHist(image_uint8)

    # augmentation methods

    def rand_adjust_contrast(self, image_array):
        img = torch.tensor(image_array, dtype=torch.float32)
        transform = RandAdjustContrast(prob=1.0, gamma=1.5)
        return transform(img).numpy()

    def rand_gaussian_smooth(self, image_array):
        img = torch.tensor(image_array, dtype=torch.float32)
        transform = RandGaussianSmooth(prob=1.0)
        return transform(img).numpy()

    def rand_flip(self, image_array):
        img = torch.tensor(image_array, dtype=torch.float32)
        transform = RandFlip(prob=0.5, spatial_axis=0)
        return transform(img).numpy()

    def rand_zoom(self, image_array):
        img = torch.tensor(image_array, dtype=torch.float32)
        transform = RandZoom(min_zoom=0.7, max_zoom=1.3, prob=1.0, mode='bilinear')
        return transform(img).numpy()
    
    def rand_rotate(self, image_array):
        # ensure correct shape
        img = np.array(image_array)
        if img.ndim == 2:
            img = img[np.newaxis, ...]  # (H, W) -> (1, H, W)
        
        img_tensor = torch.tensor(img, dtype=torch.float32)
        transform = RandRotate(range_x=np.pi/12, prob=1.0, mode='bilinear')
        rotated = transform(img_tensor).numpy()
        
        # squeeze to remove channel dimension for visualization
        return np.squeeze(rotated)
    
    def apply_technique(self, image_array, technique_name):  
        return self.techniques[technique_name](image_array)

In [82]:
def compare_images(sample_idx, technique_name):
    preprocessor = CTPreprocessingPipeline()
    image, label = train_dataset[sample_idx]
    image_array = np.array(image)
    
    processed_image = preprocessor.apply_technique(image_array, technique_name)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    axes[0].imshow(image_array, cmap='gray')
    axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(processed_image, cmap='gray')
    axes[1].set_title(f'Processed: {technique_name}', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'output/image/preprocessing/{technique_name}_sample{sample_idx}.png', bbox_inches='tight', dpi=300)
    plt.close()

In [ ]:
os.makedirs('output/image/preprocessing', exist_ok=True)

compare_images(0, 'scale_intensity')
compare_images(0, 'resize')
compare_images(0, 'clahe')
compare_images(0, 'histogram_equalization')
compare_images(0, 'rand_adjust_contrast')
compare_images(0, 'rand_gaussian_smooth')
compare_images(0, 'rand_flip')
compare_images(0, 'rand_zoom')
compare_images(0, 'rand_rotate')
compare_images(0, 'gaussian_smooth')
compare_images(0, 'sobel_edges')
compare_images(0, 'gaussian_sharpen')
compare_images(0, 'median_filter')